# F5-probability — Session 05: Hoeffding's Inequality

**Session length:** about 80 minutes • **Concept:** hoeffding-inequality.

This session turns “the sample mean should be close” into a finite-sample guarantee. We track every assumption, interval width, tail constant, union-bound step, and vacuous bound. Checkpoint answers are collected at the end.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 1. From empirical evidence to a guarantee

For independent $X_1,\ldots,X_n$, let

$$\bar X=\frac1n\sum_iX_i,\qquad E[\bar X]=\frac1n\sum_iE[X_i].$$

A simulation shows what happened in selected repetitions. A concentration inequality bounds the probability of events such as $\bar X-E\bar X\ge\varepsilon$ across the random experiment.


In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
trials = rng.integers(0, 2, size=(20_000, 40))
means = trials.mean(axis=1)
print("empirical two-sided tail:", (np.abs(means - 0.5) >= 0.15).mean())


### Checkpoint 1

Which axis represents repetitions, and which holds the 40 variables in one repetition? Why is the printed frequency evidence rather than a theorem?


## 2. Assumptions and range widths

Hoeffding requires independent variables and known almost-sure bounds $a_i\le X_i\le b_i$. Define $w_i=b_i-a_i$, and require $\varepsilon>0$. The variables need not be identically distributed or share a mean.

For Bernoulli variables, $w_i=1$. If $Y_i=3X_i-2$ for Bernoulli $X_i$, then $Y_i\in[-2,1]$ and its width is 3—not its upper endpoint.

### Checkpoint 2

Give the widths of intervals $[-1,2]$, $[4,4.5]$, and $[0,10]$. Why do observed sample minima and maxima not prove almost-sure bounds?


## 3. One-sided Hoeffding

Under those assumptions,

$$P(\bar X-E\bar X\ge\varepsilon)\le
\exp\left(-\frac{2n^2\varepsilon^2}{\sum_i(b_i-a_i)^2}\right).$$

The lower tail has the same bound. This is about deviation from $E\bar X$, not necessarily a shared mean. Track squared widths, $n^2$ in the heterogeneous form, and no leading 2 for one tail.

### Checkpoint 3

For three independent variables bounded in $[0,1]$, $[-1,1]$, and $[2,5]$, write the full upper-tail bound at $\varepsilon=0.4$.


## 4. Equal widths and vacuous bounds

If all widths equal $w$, then $\sum_iw_i^2=nw^2$ and

$$P(\bar X-E\bar X\ge\varepsilon)\le
\exp\left(-\frac{2n\varepsilon^2}{w^2}\right).$$

A raw two-sided expression can exceed 1. It remains true but is **vacuous**. Preserve the theorem's raw expression when requested; for a useful probability envelope report `min(1.0, raw_bound)`.


In [ ]:
def hoeffding_one_sided(n, epsilon, width):
    if n <= 0 or epsilon <= 0 or width <= 0:
        raise ValueError("n, epsilon, and width must be positive")
    return np.exp(-2.0 * n * epsilon**2 / width**2)

print("one-sided:", hoeffding_one_sided(40, 0.15, 1.0))


### Checkpoint 4

What happens to the exponent if every width doubles? If a raw two-sided expression is 1.4, is the inequality false, and what useful envelope should be reported?


## 5. Event subadditivity: the union bound

**This is the immediate prerequisite for the two-sided constant.** For any events $A,B$,

$$P(A\cup B)\le P(A)+P(B).$$

Indeed, $A\cup B$ is the disjoint union of $A$ and $B\setminus A$, so
$P(A\cup B)=P(A)+P(B\setminus A)\le P(A)+P(B)$. No independence is needed.

### Checkpoint 5

If $P(A)=0.2$, $P(B)=0.3$, and $P(A\cap B)=0.1$, compute both sides and identify the double-counted mass.


## 6. The two-sided factor 2

Immediately apply the lemma. Define

$$U=\{\bar X-E\bar X\ge\varepsilon\},\qquad
L=\{E\bar X-\bar X\ge\varepsilon\}.$$

Then

$$\begin{aligned}
P(|\bar X-E\bar X|\ge\varepsilon)
&=P(U\cup L)\\
&\le P(U)+P(L)\\
&\le 2\exp\left(-\frac{2n^2\varepsilon^2}{\sum_i(b_i-a_i)^2}\right).
\end{aligned}$$

For common width $w$, this is $2\exp(-2n\varepsilon^2/w^2)$. The leading 2 comes from adding upper and lower bounds—not from independence of $U$ and $L$.

### Checkpoint 6

Reproduce the three lines, naming $U$ and $L$, and mark exactly where the factor 2 appears.


## 7. Worked exam-style normal form

Twelve independent measurements have six intervals of width 1 and six of width 2. For $\varepsilon=1/2$,

$$\sum_iw_i^2=6(1^2)+6(2^2)=30,$$

so the two-sided bound is

$$2\exp\left(-\frac{2(12)^2(1/2)^2}{30}\right)=2e^{-12/5}.$$

It is below 1 and therefore informative.


In [ ]:
n = 12
epsilon = 0.5
widths = np.array([1.0] * 6 + [2.0] * 6)
raw_bound = 2.0 * np.exp(-2.0 * n**2 * epsilon**2 / np.sum(widths**2))
print("raw:", raw_bound, "useful:", min(1.0, raw_bound))


### Checkpoint 7

Why is $(\sum_iw_i)^2$ the wrong denominator? Compute it and $\sum_iw_i^2$ here; say whether the wrong expression is stronger or weaker.


## 8. Simulation versus the theoretical envelope

A fixed-seed empirical frequency is one reproducible estimate. Hoeffding is a distribution-free upper bound justified by independence and known ranges; it is often conservative.


In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
n_trials, n, p, epsilon = 50_000, 40, 0.35, 0.15
sample_means = rng.binomial(1, p, size=(n_trials, n)).mean(axis=1)
empirical_tail = (np.abs(sample_means - p) >= epsilon).mean()
raw_bound = 2.0 * np.exp(-2.0 * n * epsilon**2)
print("empirical:", empirical_tail)
print("raw Hoeffding:", raw_bound, "useful:", min(1.0, raw_bound))


**Common pitfalls:** dependence; treating observed extrema as bounds; endpoints instead of widths; unsquared widths; wrong $n$ power; missing the two-sided 2 or adding it to one tail; calling a bound above 1 false; calling simulation proof.

### Checkpoint 8

Classify those as assumption, formula, or interpretation errors. Why may an empirical tail lie far below the envelope?


## 9. Exam connections and going deeper

For Round 1 normal form: list independence, intervals, widths, side count, and threshold; state the theorem; substitute; then simplify and discuss vacuity. If asked for the factor 2, name the upper/lower events and invoke event subadditivity.

**Going deeper:** later learning theory uses union bounds for several bad events. This two-event derivation is the prerequisite prototype.


## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>
Rows are repetitions and columns are the 40 variables. A finite seeded frequency concerns those draws only.
</details>

<details><summary><b>Checkpoint 2</b></summary>
Widths: 3, 0.5, 10. Samples can miss rare extremes, so empirical extrema do not establish bounds.
</details>

<details><summary><b>Checkpoint 3</b></summary>
The squared-width sum is $1^2+2^2+3^2=14$, giving $\exp[-2(3)^2(0.4)^2/14]$.
</details>

<details><summary><b>Checkpoint 4</b></summary>
Doubling widths multiplies the denominator by 4 and quarters the exponent magnitude. A raw 1.4 is valid but vacuous; the useful envelope is 1.
</details>

<details><summary><b>Checkpoint 5</b></summary>
$P(A\cup B)=0.4\le0.5$; the intersection mass 0.1 is double-counted on the right.
</details>

<details><summary><b>Checkpoint 6</b></summary>
$P(|\cdot|\ge\varepsilon)=P(U\cup L)\le P(U)+P(L)\le e^{-K}+e^{-K}=2e^{-K}$. The final addition creates 2.
</details>

<details><summary><b>Checkpoint 7</b></summary>
Hoeffding uses $\sum_iw_i^2=30$, not $(\sum_iw_i)^2=18^2=324$. The wrong larger denominator yields a weaker, larger bound.
</details>

<details><summary><b>Checkpoint 8</b></summary>
Dependence/unproved ranges are assumption errors; widths, squares, $n$, and side constants are formula errors; vacuity and simulation-as-proof are interpretation errors. An upper bound need not be attained.
</details>
